In [131]:
from pathlib import Path
from tribo_2D import utilities
import subprocess
from ase import io
import numpy as np
import re
x = 100
y = 100
filename = f"h-mos2_1.lmp"
cif_path = "../tribo_2D/cif/h-MoS2.cif"
pot_path = "../tribo_2D/Potentials/sw_lammps/h-MoS2.sw"
multiplier = utilities.check_potential_cif_compatibility(cif_path, pot_path)
potentials = utilities.count_atomtypes(pot_path)
typecount = sum(potentials.values())
print(potentials)
print(typecount)

{'Mo': 4, 'S': 8}
12


In [132]:
def renumber_atom_types(filename, pot=None):
    """
    Renumber atom types in a LAMMPS data file to match the order and types specified in the given potential.

    Args:
        pot (list of str): List of atom type names in the desired order, typically as specified in the potential file.
        filename (str): Path to the LAMMPS data file to be modified.

    Notes:
        - The function assumes that the LAMMPS data file contains "Masses" and "Atoms" sections.
        - The function modifies the file in-place, overwriting the original file.
        - Atom type names are matched using comments (e.g., `# C`) in the "Masses" section.
        - The function may require adaptation for files with different formatting.
    """
    # Open the LAMMPS data file and read all lines
    with open(filename, 'r') as f:
        lines = f.readlines()

    masses_section = False
    atoms_section = False

    atom_types = {}
    # Parse the "Masses" section to extract atom type IDs, masses, and names
    for i, line in enumerate(lines):
        if line.strip() == 'Masses':
            masses_section = True
            continue

        if masses_section:
            if 'Atoms' in line:
                break

            parts = line.split()
            if len(parts) < 2:
                continue

            atom_type_id = int(parts[0])
            mass = float(parts[1])
            if '#' in line:
                atom_type_name = line.split('#')[-1].strip()
                lines[i] = ''
            else:
                atom_type_name = f'Unknown_{atom_type_id}'
            atom_types[atom_type_id] = (atom_type_name, mass)

    # Prepare to update atom types in the "Atoms" section
    modified_lines = set()
    mod_lines = {}
    elem_pot = {}
    elem = {}

    t_add = len(atom_types) if pot is not None else 1
    # For each atom type, update its ID and collect info
    for i in range(1, len(atom_types)+1):
        atoms_section = False
        t = i
        for l, line in enumerate(lines):
            stripped_line = line.strip()

            if 'Atoms' in line:
                atoms_section = True
                continue

            if atoms_section and stripped_line and l not in modified_lines:
                parts = stripped_line.split()

                if parts[1] == str(i):
                    parts[1] = parts[0] = str(t)
                    lines[l] = ''
                    mod_lines[t] = '  '.join(parts) + '\n'
                    modified_lines.add(l)
                    elem[t] = atom_types[i]
                    t += t_add

    a = 1
    atom_lines = {}
    # Assign new atom type numbers according to the order in 'pot'
    if pot is not None:
        for i in pot:
            atoms_section = False
            for l in range(1, len(mod_lines)+1):
                stripped_line = mod_lines[l].strip()
                parts = stripped_line.split()
                if elem[int(parts[1])][0] == i:
                    elem_pot[a] = elem[int(parts[1])]
                    parts[1] = str(a)
                    atom_lines[a] = '  '.join(parts) + '\n'
                    a += 1
        elem = elem_pot
        mod_lines = atom_lines

    # Update the number of atom types in the header and rewrite the "Masses" section
    masses_section = False
    for i, line in enumerate(lines):
        if re.match(r'^\s*\d+\s+atom types\s*$', line.strip()):
            lines[i] = f"  {len(elem)}  atom types\n"
            continue

        if line.strip() == 'Masses':
            masses_section = True
            continue

        if masses_section:
            for l in range(1, len(elem)+1):
                lines[i] += f"{l} {elem[l][1]}  #{elem[l][0]}\n"
            break
    # Write the updated lines back to the file
    with open(filename, 'w') as f:
        f.writelines(lines)
    # Append the updated atom lines at the end of the file
    with open(filename, 'a') as f:
        for line in mod_lines.values():
            f.write(line)

In [133]:
Path(filename).unlink(missing_ok=True)

# Generate the 2D sheet using Atomsk
atomsk_command = f"echo n | atomsk {cif_path} -ow {filename} -v 0"
subprocess.run(atomsk_command, shell=True, check=True)

if any(v != 1 for v in potentials) or multiplier != 1:# Open the LAMMPS data file and read all lines
    renumber_atom_types(filename)


In [134]:
atomsk_command = f"atomsk {filename} -orthogonal-cell -ow lmp -v 0"
subprocess.run(atomsk_command, shell=True, check=True)
# Duplicate atoms to match the number of types in the potential file if necessary
if multiplier != 1:
    atoms = io.read(filename, format="lammps-data")
    natoms = len(atoms)
    if typecount % natoms == 0:
        for i in range(int(np.sqrt(typecount/natoms))+1, 0, -1):
            if typecount/natoms % i == 0:
                a = int(i)
                b = int(typecount / natoms / i)
                break
    atomsk_command = f"echo n | atomsk {filename} -duplicate {a} {b} 1 -ow lmp -v 0"
    subprocess.run(atomsk_command, shell=True, check=True)
    renumber_atom_types(filename, pot=potentials)


In [130]:
# Duplicate the sheet to match the specified dimensions
dim = utilities.get_model_dimensions(filename)
duplicate_a = round(x / dim['xhi'])
duplicate_b = round(y / dim['yhi'])
atomsk_command = f"atomsk {filename} -duplicate {duplicate_a} {duplicate_b} 1 -center com -ow lmp -v 0"
subprocess.run(atomsk_command, shell=True, check=True)

CompletedProcess(args='atomsk h-mos2_1.lmp -duplicate 16 18 1 -center com -ow lmp -v 0', returncode=0)